# Multi-Fidelity Gaussian Process

Combine cheap low-fidelity and expensive high-fidelity data.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.models.gp.multifidelity import MultiFidelityGP

## Generate Multi-Fidelity Data

In [ ]:
# True function
def f_true(x):
    return np.sin(2 * x)

# Low-fidelity: biased approximation (50 points)
np.random.seed(42)
X_lo = np.linspace(0, 2 * np.pi, 50).reshape(-1, 1)
y_lo = np.sin(2 * X_lo) + 0.5 * np.cos(X_lo) + np.random.randn(*X_lo.shape) * 0.05

# High-fidelity: accurate but expensive (10 points)
X_hi = np.linspace(0.5, 5.5, 10).reshape(-1, 1)
y_hi = f_true(X_hi) + np.random.randn(*X_hi.shape) * 0.01

print(f"Low-fidelity data: {X_lo.shape[0]} points")
print(f"High-fidelity data: {X_hi.shape[0]} points")

## Fit Multi-Fidelity GP

In [ ]:
# Create and fit Multi-Fidelity GP
model = MultiFidelityGP()
model.fit(X_lo, y_lo, X_hi, y_hi)
model.optimize(n_iter=100)

## Predict

In [ ]:
# Test points
X_test = np.linspace(0, 2 * np.pi, 200).reshape(-1, 1)

# Predict with uncertainty for both fidelities
mean_lo, std_lo = model.predict_uq(X_test, fidelity="low")
mean_hi, std_hi = model.predict_uq(X_test, fidelity="high")

print(f"High-fidelity mean std: {std_hi.mean():.4f}")
print(f"Low-fidelity mean std: {std_lo.mean():.4f}")

## Visualize

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# True function
ax.plot(X_test.ravel(), f_true(X_test).ravel(), "k--", label="True function", linewidth=2)

# Low-fidelity GP
ax.fill_between(X_test.ravel(), (mean_lo - 2*std_lo).ravel(), (mean_lo + 2*std_lo).ravel(),
                alpha=0.15, color="blue", label="Low-fidelity ± 2σ")
ax.plot(X_test.ravel(), mean_lo.ravel(), "b-", alpha=0.5, label="Low-fidelity mean")

# High-fidelity GP
ax.fill_between(X_test.ravel(), (mean_hi - 2*std_hi).ravel(), (mean_hi + 2*std_hi).ravel(),
                alpha=0.3, color="red", label="High-fidelity ± 2σ")
ax.plot(X_test.ravel(), mean_hi.ravel(), "r-", linewidth=2, label="High-fidelity mean")

# Data points
ax.scatter(X_lo.ravel(), y_lo.ravel(), c="blue", s=20, alpha=0.5, label="Low-fidelity data")
ax.scatter(X_hi.ravel(), y_hi.ravel(), c="red", s=80, zorder=5, label="High-fidelity data")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Multi-Fidelity GP: Tighter intervals near high-fidelity observations")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()